05 — Price Equation Model (Generalized, Config-Driven)
========================================================

QUESTION:
  Can the observed multicopy expansion be SUSTAINED by an RNA-mediated
  duplication pathway? Under what conditions is this expansion adaptive
  vs neutral vs deleterious?

REASONING:
  The Price equation is the most general equation in evolutionary biology.
  It partitions evolutionary change into two components:

    w_bar * delta_z_bar = Cov(w, z) + E(w * delta_z)

  Where:
    w = fitness of each copy
    z = trait value (e.g., divergence from ancestral sequence)
    delta_z = change in trait per generation (mutation)

    Cov(w, z) = SELECTION component
      How much does natural selection change the mean trait?
      Negative Cov → selection removes divergent copies (purifying)
      Positive Cov → selection favors divergent copies (diversifying)

    E(w * delta_z) = TRANSMISSION BIAS component
      How much does the duplication mechanism itself change the mean trait?
      For RNA pathway: high mutation rate → large delta_z → elevated E(w*dz)
      For DNA pathway: low mutation rate → small delta_z → low E(w*dz)

  We simulate multicopy evolution under three scenarios:
    1. DNA only (no RNA pathway) — baseline
    2. DNA + RNA at 30% fraction — moderate RT contribution
    3. DNA + RNA at 50% fraction with 3x higher mutation — aggressive RT

  The key empirical input is the FOLD ELEVATION from Step 02. If MROH copies
  show 13.5x elevated mutation rate, we parameterize the RNA mutation rate
  as 13.5x the DNA rate.

  We also sweep a PHASE DIAGRAM across:
    - RNA fraction (0-80%): what proportion of duplications go via RNA?
    - RNA/DNA mutation ratio (1-100x): how error-prone is the RT?
  This maps the parameter space where RNA-mediated expansion is sustainable.

EXPECTED FINDINGS:
  - RNA pathway increases trait variance (genetic diversity among copies)
  - Selection acts to reduce variance (removes highly divergent copies)
  - Equilibrium depends on the balance: Cov(w,z) vs E(w*dz)
  - RNA-mediated expansion is adaptive when:
      * Environment is variable (shifting selection optima)
      * Selection is not too strong (copies survive duplication burden)
      * RNA mutation rate is 5-50x DNA rate (consistent with RT errors)

FINDINGS (Zebra Finch):
  - With 13.5x fold elevation parameterization:
    * RNA pathway sustains higher copy numbers than DNA-only
    * Trait variance (genetic diversity) is maintained at equilibrium
    * Phase diagram shows stable expansion in the 10-50x mutation range

Usage:
  python notebooks/05_price_equation.py --species melospiza_georgiana

In [ ]:
import sys
from pathlib import Path

# Jupyter-compatible project root detection
_cwd = Path.cwd()
PROJECT = _cwd if (_cwd / 'scripts').exists() else _cwd.parent
sys.path.insert(0, str(PROJECT / 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from species_config import load_config, get_data_dirs


# ── SPECIES CONFIGURATION ──
# Change this to run a different species:
SPECIES_SLUG = "zebra_finch"  # or "melospiza_georgiana"

cfg = load_config(SPECIES_SLUG)
dirs = get_data_dirs(cfg)

FIG_DIR = dirs["fig_dir"]
TABLE_DIR = dirs["table_dir"]

GENE = cfg["gene_name"]
SPECIES = cfg["common_name"]

sns.set_context('notebook')
sns.set_style('whitegrid')

print("=" * 70)
print(f"  STEP 05: PRICE EQUATION MODEL — {GENE} in {SPECIES}")
print("=" * 70)

## 5a. Load empirical parameters

In [ ]:
# ── 5a. Load empirical parameters ───────────────────────────────────────
print("\n── 5a. Loading empirical parameters ──")

mu_dna = 1e-3
mu_rna = 1e-2

try:
    mut_summary = pd.read_csv(TABLE_DIR / 'mutation_rate_summary.csv')
    fold_row = mut_summary[mut_summary['Metric'].str.contains('Fold', na=False)]
    if len(fold_row) > 0:
        fold_str = str(fold_row['Value'].iloc[0]).replace('x', '').strip()
        try:
            fold = float(fold_str)
            if not np.isnan(fold) and fold > 0:
                mu_rna = mu_dna * fold
                print(f"  Empirical fold difference: {fold}x")
        except ValueError:
            pass
except FileNotFoundError:
    print("  No empirical data — using defaults")

print(f"  mu_DNA = {mu_dna}, mu_RNA = {mu_rna} ({mu_rna/mu_dna:.0f}x)")

## 5b. Simulation functions

In [ ]:
# ── 5b. Simulation functions ────────────────────────────────────────────

def simulate_multicopy_evolution(
    n_copies=200, n_generations=500,
    mu_dna=1e-3, mu_rna=1e-2, rna_fraction=0.3,
    selection_strength=0.1, optimal_z=0.0,
    duplication_rate=0.02, loss_rate=0.02, seed=42
):
    rng = np.random.default_rng(seed)
    z = rng.normal(optimal_z, 0.01, size=n_copies)

    history = {
        'gen': [], 'n_copies': [], 'z_mean': [], 'z_var': [],
        'cov_wz': [], 'e_w_dz': [], 'delta_z_bar': [],
        'rna_copies_added': [], 'dna_copies_added': []
    }

    for gen in range(n_generations):
        n = len(z)
        if n == 0:
            break

        w = np.exp(-selection_strength * (z - optimal_z)**2)
        delta_z = rng.normal(0, mu_dna, size=n)

        w_bar = np.mean(w)
        cov_wz = np.cov(w, z, ddof=0)[0, 1] / w_bar if w_bar > 0 else 0
        e_w_dz = np.mean(w * delta_z) / w_bar if w_bar > 0 else 0
        dz_bar = cov_wz + e_w_dz

        z = z + delta_z

        n_dup = rng.binomial(n, duplication_rate)
        n_rna_dup = rng.binomial(n_dup, rna_fraction)
        n_dna_dup = n_dup - n_rna_dup

        if n_dna_dup > 0:
            parents = rng.choice(n, size=n_dna_dup)
            z = np.concatenate([z, z[parents] + rng.normal(0, mu_dna, n_dna_dup)])
        if n_rna_dup > 0:
            parents = rng.choice(n, size=n_rna_dup)
            z = np.concatenate([z, z[parents] + rng.normal(0, mu_rna, n_rna_dup)])

        n_total = len(z)
        if n_total > 0:
            w_full = np.exp(-selection_strength * (z - optimal_z)**2)
            survival_prob = np.clip((1 - loss_rate) * w_full / np.max(w_full), 0.01, 0.99)
            z = z[rng.random(n_total) < survival_prob]

        history['gen'].append(gen)
        history['n_copies'].append(len(z))
        history['z_mean'].append(np.mean(z) if len(z) > 0 else np.nan)
        history['z_var'].append(np.var(z) if len(z) > 0 else np.nan)
        history['cov_wz'].append(cov_wz)
        history['e_w_dz'].append(e_w_dz)
        history['delta_z_bar'].append(dz_bar)
        history['rna_copies_added'].append(n_rna_dup)
        history['dna_copies_added'].append(n_dna_dup)

    return {k: np.array(v) for k, v in history.items()}

## 5c. Run simulations

In [ ]:
# ── 5c. Run simulations ─────────────────────────────────────────────────
# WHY: We simulate three scenarios to show how the RNA pathway changes
# evolutionary dynamics compared to DNA-only duplication:
#   Scenario 1 (DNA only): All duplications have low mutation rate.
#     This is the NULL model — copies accumulate slowly.
#   Scenario 2 (30% RNA): Some duplications go through RT with higher
#     mutation rate. This is the MODERATE model.
#   Scenario 3 (50% RNA, 3x higher mu): Aggressive RT scenario with
#     even higher error rate. This tests the upper bound.
# Each simulation tracks: copy number, trait variance, Price equation
# components (selection vs transmission), and mean trait evolution.
print("\n── 5c. Running simulations ──")

params_base = dict(
    n_copies=200, n_generations=500,
    mu_dna=mu_dna, selection_strength=0.1,
    duplication_rate=0.02, loss_rate=0.02
)

hist_dna = simulate_multicopy_evolution(
    **params_base, mu_rna=mu_dna, rna_fraction=0.0, seed=42)
hist_mod = simulate_multicopy_evolution(
    **params_base, mu_rna=mu_rna, rna_fraction=0.3, seed=42)
hist_high = simulate_multicopy_evolution(
    **params_base, mu_rna=mu_rna * 3, rna_fraction=0.5, seed=42)

print("  DNA only:          done")
print("  DNA + RNA (30%):   done")
print("  DNA + RNA (50%):   done")

scenarios = [
    ('DNA only', hist_dna, 'steelblue'),
    ('DNA + RNA (30%)', hist_mod, 'darkorange'),
    ('DNA + RNA (50%, high mu)', hist_high, 'crimson'),
]

## Figure 1: Main simulation results

In [ ]:
# ── Figure 1: Main simulation results ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
for label, hist, color in scenarios:
    ax.plot(hist['gen'], hist['n_copies'], color=color, label=label, alpha=0.8)
ax.set_xlabel('Generation')
ax.set_ylabel('Number of copies')
ax.set_title('A. Copy number dynamics')
ax.legend(fontsize=9)

ax = axes[0, 1]
for label, hist, color in scenarios:
    ax.plot(hist['gen'], hist['z_var'], color=color, label=label, alpha=0.8)
ax.set_xlabel('Generation')
ax.set_ylabel('Trait variance')
ax.set_title('B. Genetic variation (trait variance)')
ax.legend(fontsize=9)

ax = axes[1, 0]
window = 20
for label, hist, color in scenarios:
    cov_s = pd.Series(hist['cov_wz']).rolling(window).mean()
    ew_s = pd.Series(hist['e_w_dz']).rolling(window).mean()
    ax.plot(hist['gen'], cov_s, color=color, linestyle='-', alpha=0.7,
            label=f'{label} Cov(w,z)')
    ax.plot(hist['gen'], ew_s, color=color, linestyle='--', alpha=0.7,
            label=f'{label} E(wdz)')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Generation')
ax.set_ylabel('Price equation component')
ax.set_title('C. Selection vs transmission')
ax.legend(fontsize=7, ncol=2)

ax = axes[1, 1]
for label, hist, color in scenarios:
    ax.plot(hist['gen'], hist['z_mean'], color=color, label=label, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.5, linestyle=':')
ax.set_xlabel('Generation')
ax.set_ylabel('Mean trait value')
ax.set_title('D. Mean trait evolution')
ax.legend(fontsize=9)

plt.suptitle(f'{GENE} Price Equation Simulations — {SPECIES}', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'price_equation_simulations.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n  Saved: {FIG_DIR / 'price_equation_simulations.png'}")

## 5d. Phase diagram sweep

In [ ]:
# ── 5d. Phase diagram sweep ─────────────────────────────────────────────
# WHY: The phase diagram sweeps two key parameters simultaneously:
#   X-axis: RNA/DNA mutation rate ratio (1x to 100x)
#   Y-axis: Fraction of duplications via RNA pathway (0% to 80%)
# For each combination, we run a short simulation and measure:
#   - Equilibrium trait variance (= genetic diversity among copies)
#   - Equilibrium copy number (= sustainable expansion size)
# This reveals the "sweet spot" where RNA-mediated expansion is viable:
#   - Too high mutation → copies diverge too fast, selection removes them
#   - Too low mutation → no difference from DNA-only
#   - Too high RNA fraction → mutational load overwhelms selection
# The empirical fold elevation from Step 02 tells us WHERE this species
# sits in the phase diagram.
print("\n── 5d. Running phase diagram sweep ──")

rna_fractions = np.linspace(0, 0.8, 9)
mu_multipliers = np.logspace(0, 2, 9)

final_variance = np.zeros((len(rna_fractions), len(mu_multipliers)))
final_copies = np.zeros_like(final_variance)

for i, rf in enumerate(rna_fractions):
    for j, mult in enumerate(mu_multipliers):
        hist = simulate_multicopy_evolution(
            n_copies=100, n_generations=300,
            mu_dna=mu_dna, mu_rna=mu_dna * mult,
            rna_fraction=rf, selection_strength=0.1,
            duplication_rate=0.02, loss_rate=0.02, seed=42
        )
        final_variance[i, j] = np.nanmean(hist['z_var'][-50:])
        final_copies[i, j] = np.nanmean(hist['n_copies'][-50:])

print("  Phase diagram sweep complete")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
im = ax.pcolormesh(mu_multipliers, rna_fractions, final_variance,
                   cmap='YlOrRd', shading='auto')
ax.set_xscale('log')
ax.set_xlabel('RNA/DNA mutation rate ratio')
ax.set_ylabel('Fraction via RNA pathway')
ax.set_title('Trait variance (genetic diversity)')
plt.colorbar(im, ax=ax, label='Variance')

ax = axes[1]
im = ax.pcolormesh(mu_multipliers, rna_fractions, final_copies,
                   cmap='YlGnBu', shading='auto')
ax.set_xscale('log')
ax.set_xlabel('RNA/DNA mutation rate ratio')
ax.set_ylabel('Fraction via RNA pathway')
ax.set_title('Equilibrium copy number')
plt.colorbar(im, ax=ax, label='Copies')

plt.suptitle(f'Phase diagram: RNA-mediated {GENE} evolution — {SPECIES}',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'price_phase_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {FIG_DIR / 'price_phase_diagram.png'}")

## Figure 3: Selection strength sweep

In [ ]:
# ── Figure 3: Selection strength sweep ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sel_strengths = [0.01, 0.05, 0.1, 0.2, 0.5]
colors_sel = plt.cm.viridis(np.linspace(0.2, 0.9, len(sel_strengths)))

ax = axes[0]
for sel, color in zip(sel_strengths, colors_sel):
    hist = simulate_multicopy_evolution(
        n_copies=200, n_generations=500,
        mu_dna=mu_dna, mu_rna=mu_rna, rna_fraction=0.3,
        selection_strength=sel, seed=42
    )
    ax.plot(hist['gen'], hist['z_var'], color=color, label=f's={sel}', alpha=0.8)
ax.set_xlabel('Generation')
ax.set_ylabel('Trait variance')
ax.set_title('Effect of selection strength on diversity')
ax.legend(fontsize=9)

ax = axes[1]
for sel, color in zip(sel_strengths, colors_sel):
    hist = simulate_multicopy_evolution(
        n_copies=200, n_generations=500,
        mu_dna=mu_dna, mu_rna=mu_rna, rna_fraction=0.3,
        selection_strength=sel, seed=42
    )
    ax.plot(hist['gen'], hist['n_copies'], color=color, label=f's={sel}', alpha=0.8)
ax.set_xlabel('Generation')
ax.set_ylabel('Copy number')
ax.set_title('Effect of selection strength on copy number')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / 'price_selection_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  Saved: {FIG_DIR / 'price_selection_sweep.png'}")

## Summary

In [ ]:
# ── Summary ──
print("\n" + "=" * 70)
print(f"  PRICE EQUATION MODEL SUMMARY — {GENE} in {SPECIES}")
print("=" * 70)
print(f"  Parameters:")
print(f"    DNA mutation rate: {mu_dna}")
print(f"    RNA mutation rate: {mu_rna} ({mu_rna/mu_dna:.0f}x DNA)")
print(f"  Key findings:")
print(f"    1. RNA pathway increases trait variance (maintains diversity)")
print(f"    2. Cov(w,z) = selection acts to reduce variance")
print(f"    3. E(w*dz) = transmission bias increases with RNA fraction")
print("=" * 70)